# SeaDronesSee v2 — Faster R-CNN ConvNeXt-FPN Selective QAT
Pipeline dành cho Google Colab: tải riêng bản JPEG compressed (~10 GB), train FP32, fine-tune QAT M3 và lưu checkpoint trực tiếp vào Google Drive. Chọn **Runtime → Change runtime type → GPU** trước khi chạy.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
assert torch.cuda.is_available(), 'Hãy bật GPU runtime trước'
print(torch.cuda.get_device_name(0))
print('GPU memory GB:', torch.cuda.get_device_properties(0).total_memory / 2**30)

## 1. Lấy source code
Commit/push các thay đổi pipeline lên GitHub trước khi chạy notebook này.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path('/content/EchteAI')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/NguyenDucThang-tb/EchteAI.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)
print('Repository:', Path.cwd())

In [ ]:
!pip -q install -e '.[coco]' psutil
!apt-get -qq update && apt-get -qq install -y rclone
!python -m compileall -q Python/EchteAI/pipelines/convnext_qat scripts

## 2. Tải dataset vào SSD tạm của Colab
Không đặt dataset 10 GB trên Drive vì đọc hàng nghìn ảnh qua Drive rất chậm. Checkpoint vẫn được ghi vào Drive. `rclone copy` có thể chạy lại và sẽ tiếp tục các file còn thiếu.

In [ ]:
DATA_ROOT = Path('/content/seadronessee')
DAV_URL = 'https://cloud.cs.uni-tuebingen.de/public.php/dav/files/ZZxX65FGnQ8zjBP/'
subprocess.run(['rclone', 'config', 'delete', 'seadrones'], check=False, capture_output=True)
subprocess.run(['rclone', 'config', 'create', 'seadrones', 'webdav', 'url', DAV_URL, 'vendor', 'nextcloud'], check=True)
subprocess.run([
    'rclone', 'copy', 'seadrones:Compressed Version', str(DATA_ROOT),
    '--progress', '--transfers', '8', '--checkers', '16', '--retries', '10',
], check=True)
required = [
    DATA_ROOT/'images/train', DATA_ROOT/'images/val', DATA_ROOT/'images/test',
    DATA_ROOT/'annotations/instances_train.json',
    DATA_ROOT/'annotations/instances_val.json',
]
assert all(path.exists() for path in required), [str(path) for path in required if not path.exists()]
print('Dataset ready at', DATA_ROOT)

In [ ]:
import json, collections
for split in ('train', 'val'):
    data = json.loads((DATA_ROOT/f'annotations/instances_{split}.json').read_text())
    counts = collections.Counter(a['category_id'] for a in data['annotations'])
    names = {c['id']: c['name'] for c in data['categories']}
    print(split, 'images=', len(data['images']), 'boxes=', len(data['annotations']))
    print({names[key]: value for key, value in counts.items()})

## 3. Smoke test

In [ ]:
!python tests/smoke_selective_qat.py

## 4. Train FP32 baseline
Checkpoint tốt nhất và checkpoint cuối mỗi epoch đều nằm trong Drive. Cell tự resume từ `fp32_last.pt` nếu Colab đã bị ngắt trước đó.

In [ ]:
import datetime

def run_and_log(command, log_path):
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('Command:', ' '.join(command), flush=True)
    print('Persistent log:', log_path, flush=True)
    print('Started:', datetime.datetime.now().isoformat(timespec='seconds'), flush=True)
    with log_path.open('a', encoding='utf-8') as log_file:
        log_file.write(f'\n===== START {datetime.datetime.now().isoformat()} =====\n')
        log_file.write(' '.join(command) + '\n')
        log_file.flush()
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env=env,
        )
        for line in process.stdout:
            print(line, end='', flush=True)
            log_file.write(line)
            log_file.flush()
        return_code = process.wait()
        log_file.write(f'===== END code={return_code} {datetime.datetime.now().isoformat()} =====\n')
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

CONFIG = 'configs/seadronessee_colab.yaml'
DRIVE_OUT = Path('/content/drive/MyDrive/EchteAI/seadronessee_m3')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
fp32_last = DRIVE_OUT/'fp32_last.pt'
command = [sys.executable, '-u', 'scripts/train_fp32.py', '--config', CONFIG]
if fp32_last.exists():
    command += ['--resume', str(fp32_last)]
    print('Resume checkpoint found:', fp32_last, flush=True)
else:
    print('No resume checkpoint; starting FP32 training from epoch 0', flush=True)
run_and_log(command, DRIVE_OUT/'fp32_train.log')

## 5. Selective QAT M3
Calibration 512 ảnh → weight-only 2 epoch → full QAT 6 epoch → frozen observer 3 epoch → convert INT8. Cell tự resume từ Drive.

In [ ]:
qat_last = DRIVE_OUT/'qat_last.pt'
command = [sys.executable, '-u', 'scripts/train_qat.py', '--config', CONFIG, '--variant', 'M3']
if qat_last.exists():
    command += ['--resume', str(qat_last)]
    print('Resume checkpoint found:', qat_last, flush=True)
else:
    print('No resume checkpoint; starting QAT from calibration', flush=True)
run_and_log(command, DRIVE_OUT/'qat_train.log')

## 6. Evaluate trên validation
Test annotation chính thức bị giữ kín, vì vậy metric local được tính trên validation.

In [ ]:
subprocess.run([sys.executable, 'scripts/evaluate.py', '--config', CONFIG, '--model', 'fp32', '--split', 'val'], check=True)
subprocess.run([sys.executable, 'scripts/evaluate.py', '--config', CONFIG, '--model', 'int8', '--split', 'val'], check=True)

## 7. CPU benchmark và kiểm tra file Drive

In [ ]:
subprocess.run([sys.executable, 'scripts/benchmark.py', '--config', CONFIG], check=True)
for path in sorted(DRIVE_OUT.glob('*')):
    print(f'{path.name:28s} {path.stat().st_size / 2**20:9.2f} MB')